# Netflix Content Strategy — Dashboard Preparation

In [ ]:
!pip install -q pyspark==3.5.3


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Netflix-Content-Strategy-Dashboard-Prep")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)


In [ ]:
CSV_PATH = "/content/clean_netflix.csv"

clean = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(CSV_PATH)
    .withColumn("release_year", F.col("release_year").cast("int"))
    .withColumn("content_age", F.col("content_age").cast("double"))
)

clean.printSchema()


In [ ]:
dashboard = (
    clean
    .filter(F.col("release_year").isNotNull())
    .filter(F.col("release_year") >= 2008)
    .withColumn(
        "content_age_group",
        F.when(F.col("content_age") <= 2, "Very Recent")
         .when(F.col("content_age") <= 5, "Recent")
         .when(F.col("content_age") <= 10, "Mature")
         .when(F.col("content_age") <= 20, "Older")
         .otherwise("Legacy")
    )
    .withColumn(
        "release_decade",
        (F.floor(F.col("release_year") / 10) * 10).cast("int")
    )
    .withColumn(
        "is_recent",
        F.when(F.col("content_age") <= 5, 1).otherwise(0)
    )
)

dashboard.show(5, truncate=False)


In [ ]:
# A title-level dataset avoids double counting caused by exploded genres.
title_level = (
    dashboard
    .groupBy("show_id")
    .agg(
        F.first("title").alias("title"),
        F.first("type").alias("type"),
        F.first("primary_country").alias("primary_country"),
        F.first("release_year").alias("release_year"),
        F.first("rating").alias("rating"),
        F.first("content_age").alias("content_age"),
        F.first("content_age_group").alias("content_age_group"),
        F.first("release_decade").alias("release_decade"),
        F.first("is_recent").alias("is_recent")
    )
)

title_level.toPandas().to_csv("/content/content_title_level.csv", index=False)
dashboard.toPandas().to_csv("/content/content_strategy_extended.csv", index=False)

print("Saved dashboard datasets.")


In [ ]:
# Quick KPI validation
title_level.select(
    F.count("*").alias("total_titles"),
    F.sum(F.when(F.col("type") == "Movie", 1).otherwise(0)).alias("movies"),
    F.sum(F.when(F.col("type") == "TV Show", 1).otherwise(0)).alias("tv_shows"),
    F.round(F.avg("content_age"), 1).alias("avg_content_age"),
    F.sum("is_recent").alias("recent_titles")
).show()
